In [1]:
import torch
import torch.nn as nn

import pandas as pd
import numpy as np
import string
import re
from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt
import torch.optim as optim
from google.colab import drive

try:
    from spellchecker import SpellChecker
except:
    !pip install pyspellchecker
    from spellchecker import SpellChecker

from torch.utils.data import Dataset, DataLoader
# impport pad and pack sequence for variable length sequences
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence, pack_sequence
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA device found.", torch.cuda.is_available())
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print ("MPS device found.", torch.backends.mps.is_available())
else:
    print("Using CPU device.")

TRAIN_FROM_SCRATCH = False
SAVE_WEIGHTS = True
SPELL_CHECK = False
STRIP_PUNCT = True

ALTERNATE_DATASET = False
if ALTERNATE_DATASET:
    weight_path = "/content/drive/MyDrive/Colab Notebooks/trans_whackGPT_weights_alt.pth"
else:
    weight_path = "/content/drive/MyDrive/Colab Notebooks/trans_whackGPT_weights.pth"


CUDA device found. True


# Transformers - Robots in Disguise

We can make a transformer based model to generate chatGPT-ish text responses. Ours will be far more stupid, but hey, it's a taking computer. Transformer based models are the current state of the art for natural language processing, and most of the models that you've heard of, like ChatGPT, are transformer based models.

## Transformer Architecture

What the heck is a transformer, what does it do, and why is it so cool? A transformer model is a type of neural network that was creating in 2017 at Google. The core idea behind transformers is the idea of attention, which is detailed a little bit below. The diagrammed structure of a transformer model can be a little intimidating, but we can make sense of the critical parts without too much issue. We will mostly sidestep this large diagram, and focus on a basic transformer model that is a little easier to understand.

![Transformer](images/transformer.webp "Transformer")

A transformer model contains a few key parts, each of with is dealt with in more detail below.
<ul>
<li> Embedding - the embedding layer generates embeddings (vector representations) for each token. The embeddings are created for both the token itself and its position in the sequence. </li>
<li> Attention layers - the attention layers are the core of the transformer model. They are responsible for creating a representation of the input sequence that is used to generate the output sequence. </li>
</ul>

The attention part is the star of the show, it is a method to be able to focus the attention of the model on the critical portions of the input sequence and generate contextually informed predictions for the output. As well, transformers do all of this in a way that is more parallelizable than LSTM based models that were the state of the art before transformers, only a few years ago. 

This image comes from Google, like transformers, and is a great visual representation of how attention works. This example is from a translation model, but a more simple version applies to what we are about to do. The key thing to note is that the image shows:
<ul>
<li> Several encoding layers, each of which generates a representation of each word in the input sequence by looking at all of the others. </li>
<li> Several decoding layers, which generate a representation of the output sequence by looking at all of the words in the input sequence, as well as the current sequence being generated. </li>
</ul>

This highlights the key idea of attention, which is that the model can look at any part of the input sequence to understand the current word in the input, as well as generate the current word of the output. This differs from the LSTM models that are tied to understanding the data only as a sequence, and not as a whole. The transformer model learns to look at, or pay attention to, the important parts of the input, irrespective of their position in the sequence. This is important in language, as we can have words that impact the meaning of other words at any place in the sequence. Think of an example sentence, "he isn't the largest, fastest, strongest, or tallest, but the walk-on scrapper Rudy is the heart of the team". In this sentence "he" applies to Rudy, as does "scrappy" and "heart of the team", which is clear in English, but maybe not so easy for a machine. The transformer model is able to look at the sentence, and work to learn the relationships between the words, and the context in which they are used - so if we were generating a similar sentence, the model knows that after "scrapper" we need a person, a "he", more specifically.  

<b>Important Note:</b> these generative text models seem smart, and in some senses they are, but in the most critical sense, they really aren't, and can be confidently and totally wrong. The model understands what is written and how to construct blocks of text; the model does not understand the underlying meaning. If you ask ChatGPT what it's like to bite an apple, you'll probably get descriptors like "sweet", "tart", and "juicy", which are all accurate. The model doesn't know what it is like to bite an apple, it just knows that if it needs to supply descriptors of an object "apple", sweet, juicy, and tart are ones that commonly come up. If it was trained on text written by some apple-hating lunatic, who wrote about how mushy, bitter, and gross apples are, that's what the model will confidently state an apple is like. Even when models learn to do chemistry or math, they are only learning to replicate what they have seen - they don't understand the underlying concepts. With subject like those, there tends to be more strictly defined rules than with the English language, which is pretty loose, so it isn't surprising that these models are quick to learn those subject areas, even if it seems difficult and almost impossibly fast for us as humans - a model can learn a linear regression, from examples only, very quickly. 

In [2]:
from datasets import load_dataset

if not ALTERNATE_DATASET:
    ds = load_dataset("SocialGrep/one-million-reddit-confessions")
    text_data = ds["train"]["selftext"]
    text_data = [t for t in ds["train"]["selftext"] if not t is None]
else:
    ds = load_dataset("ysharma/short_jokes")
    text_data = ds["train"]["Joke"]
    text_data = [t for t in text_data if not t is None]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## Tokenization and Byte-Pair Encoding

We can tokenize our text and create sequences. The sequence creation is the same as LSTM text generation - previous features are the prior tokens, target is the next token. The tokenization process is a little different here, we will use byte-pair encoding, which should perform better at representing the meaning and structure of language. 

### Byte-Pair Encoding

Byte-pair encoding is a method of tokenization that is more efficient than character or word level tokenization. It works by iteratively merging the most frequent pairs of characters or tokens in the text until a desired vocabulary size is reached. This allows for a more compact representation of the text, as common words and phrases can be represented as single tokens, while still allowing for the representation of rare words and subwords. This is particularly useful for languages with a large number of unique words, such as English, where there are many infrequent words that would be inefficient to represent as individual tokens.

### Positional Encoding

Positional encoding is a method of encoding the position of each token in the input sequence. This is important for transformer models, as they do not have a built-in notion of sequence order like LSTM models do. 

The sine and cos positional encoding is a method of encoding the position of each token in the input sequence using sine and cosine functions. This method was proposed in the original transformer paper, and is based on the idea that the position of each token can be represented as a combination of sine and cosine functions of different frequencies. The idea is that the model can learn to use these functions to understand the position of each token in the input sequence, which is critical for understanding the meaning of the text. In our implementation, we will use a simple learnable positional embedding, which is added to the token embeddings before being fed into the transformer layers.


In [3]:
def byte_pair_encoding(texts, vocab_size=1000, num_merges=None):
    """
    Perform byte pair encoding on a list of texts.

    This function starts with a character-level vocabulary and iteratively
    merges the most frequent adjacent byte pair into a new vocabulary token.
    The result is a vocabulary that contains both raw bytes and merged byte pairs.
    """
    if num_merges is None:
        num_merges = vocab_size - 256
    
    # Start with the base vocabulary of all single-byte values.
    vocab = {bytes([i]): i for i in range(256)}
    
    # Convert each text into a list of byte values.
    tokens = [list(text.encode('utf-8', errors='ignore')) for text in texts]
    
    # Iteratively merge the most common adjacent byte pairs to build the BPE vocabulary.
    for i in range(num_merges):
        pairs = Counter()
        for token_list in tokens:
            for j in range(len(token_list) - 1):
                pairs[tuple(token_list[j:j+2])] += 1
        
        if not pairs:
            # Stop early if there are no adjacent pairs left to merge.
            break
        
        # Choose the most frequent byte pair to merge.
        best = pairs.most_common(1)[0][0]
        
        # Assign a new token id for the merged pair.
        new_token = len(vocab)
        vocab[best] = new_token
        
        # Replace occurrences of the chosen pair in every tokenized text.
        for token_list in tokens:
            i = 0
            while i < len(token_list) - 1:
                if tuple(token_list[i:i+2]) == best:
                    token_list[i:i+2] = [new_token]
                i += 1
    
    return tokens, vocab


def create_training_sequences(token_lists, sequence_length=50):
    """
    Create fixed-length input sequences and next-token targets for training.

    Each token list is sliced into overlapping windows of length `sequence_length`.
    The next token after each window becomes the training target.
    """
    sequences = []
    targets = []
    
    for token_list in token_lists:
        # Slide across each tokenized example to create many training examples.
        for i in range(len(token_list) - sequence_length):
            sequences.append(token_list[i:i + sequence_length])
            targets.append(token_list[i + sequence_length])
    
    # Convert the list of sequences and targets into tensors for PyTorch.
    return torch.tensor(sequences, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


In [4]:
sequence_length = 40
max_vocab_size = 10000
sample_size = 5000
if sample_size > len(text_data):
    sample_size = len(text_data)

# Apply tokenization and create sequences
tokens, vocab = byte_pair_encoding(text_data[:sample_size], vocab_size=max_vocab_size)  # Sample for efficiency
input_sequences, target_tokens = create_training_sequences(tokens, sequence_length=sequence_length)

print(f"Vocabulary size: {len(vocab)}")
print(f"Training sequences shape: {input_sequences.shape}")
print(f"Target tokens shape: {target_tokens.shape}")

Vocabulary size: 10000
Training sequences shape: torch.Size([23166, 40])
Target tokens shape: torch.Size([23166])


In [5]:
class GenerativeSequenceDataset(Dataset):
    def __init__(self, input_sequences, target_tokens):
        assert len(input_sequences) == len(target_tokens), "Inputs and targets must have same length."
        self.input_sequences = input_sequences
        self.target_tokens = target_tokens

    def __len__(self):
        return len(self.input_sequences)

    def __getitem__(self, idx):
        x = self.input_sequences[idx]
        y = self.target_tokens[idx]
        length = x.size(0)
        return x, y, length


def packed_collate_fn(batch):
    # batch: list of (x, y, length)
    xs, ys, lengths = zip(*batch)

    lengths = torch.tensor(lengths, dtype=torch.long)
    ys = torch.stack(ys)

    # sort by length (required for enforce_sorted=True)
    lengths_sorted, sort_idx = lengths.sort(descending=True)
    xs_sorted = [xs[i] for i in sort_idx]
    ys_sorted = ys[sort_idx]

    padded_x = pad_sequence(xs_sorted, batch_first=True, padding_value=0)
    packed_x = pack_padded_sequence(
        padded_x,
        lengths_sorted.cpu(),
        batch_first=True,
        enforce_sorted=True
    )

    return packed_x, ys_sorted, lengths_sorted


# Build dataset
full_dataset = GenerativeSequenceDataset(input_sequences, target_tokens)

# Train/validation split
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

# Dataloaders producing packed sequences
batch_size = 128
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=packed_collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=packed_collate_fn
)

# Example batch
packed_inputs, batch_targets, batch_lengths = next(iter(train_loader))
print("Packed input data shape:", packed_inputs.data.shape)
print("Packed batch_sizes shape:", packed_inputs.batch_sizes.shape)
print("Batch targets shape:", batch_targets.shape)
print("Batch lengths shape:", batch_lengths.shape)

Packed input data shape: torch.Size([5120])
Packed batch_sizes shape: torch.Size([40])
Batch targets shape: torch.Size([128])
Batch lengths shape: torch.Size([128])


## Transformer Models

### How this Transformer Model Works

This model is a small language model built with PyTorch using a transformer encoder. If you are comfortable with LSTM models, think of this as replacing recurrence with self-attention: instead of updating hidden state step-by-step, the transformer computes relationships across all positions in a sequence at once.

#### Model inputs and embeddings

- Input tokens are mapped with `nn.Embedding` into dense vectors.
- Positional embeddings are added to each token embedding so the model knows the order of the sequence.
- The sum of token and position embeddings is the same idea as providing an LSTM with a sequence; it gives the model a vector representation for each time step.

#### Transformer encoder and attention

- The core of the model is `nn.TransformerEncoder`, which contains stacked encoder layers.
- Each encoder layer performs multi-head self-attention followed by a feed-forward network.
- Self-attention allows every token in the input to attend to every other token in the same sequence.
- The causal mask (`torch.triu(..., diagonal=1)`) blocks future tokens, so predictions only depend on earlier context.

The training pass can be understood like this:

1. A batch of sequences enters the model.
2. Each sequence is embedded and augmented with position information.
3. The transformer computes attention weights between tokens in the sequence.
4. The final output is normalized and projected to vocabulary logits.
5. Cross-entropy loss compares the logits to the next-token targets.

#### Why attention matters

- Attention replaces the hidden state chaining used by LSTMs.
- The model can directly compare any pair of positions, so long-range dependencies are easier to capture.
- In a single encoder pass, each token gets a weighted sum of information from all previous tokens.
- The causal attention mask enforces the autoregressive property needed for language modeling.

#### Training pass example

- Input: a token sequence of length `sequence_length`.
- Target: the next token after that sequence.
- The model predicts the next-token logits from the final position output.
- Loss is computed with `nn.CrossEntropyLoss`.
- Gradients flow back through the attention and feed-forward layers, updating embeddings and projection weights.

A simplified execution diagram for training:

```text
Batch of input token IDs
        │
        ▼
Embedding + positional embedding
        │
        ▼
Transformer encoder blocks
  ┌───────────────┐
  │ self-attention│ ──> token interactions
  └───────────────┘
        │
        ▼
LayerNorm + Linear output
        │
        ▼
Logits for next token
        │
        ▼
CrossEntropyLoss vs target
        │
        ▼
Backward pass updates weights
```

#### Inference pass example

During generation, the model is used autoregressively:

1. Encode the current context tokens.
2. Run the transformer to get logits for the next token.
3. Sample or choose the next token with temperature / top-k.
4. Append that token to the context and repeat.

```text
Seed text -> token IDs
        │
        ▼
Transformer forward pass
        │
        ▼
Next-token logits
        │
        ▼
Sample token -> append to context
        │
        ▼
Repeat until max length
```

This helps connect the transformer implementation to the familiar training loop and inference pattern used in LSTM-based text generation.


In [6]:
import os
from functools import lru_cache

# ---- BPE helpers ----
# Reuse the vocabulary and merge rules computed earlier in the notebook.
# We do not retrain BPE here; this section uses the already-built `vocab`
# and `merge_rules` to convert between token ids and text during inference.
id_to_piece = {token_id: piece for piece, token_id in vocab.items()}
merge_rules = sorted(
    [(token_id, piece) for piece, token_id in vocab.items() if isinstance(piece, tuple)],
    key=lambda x: x[0]
)

@lru_cache(maxsize=None)
def token_to_bytes(token_id):
    piece = id_to_piece[token_id]
    if isinstance(piece, bytes):
        return piece
    left, right = piece
    return token_to_bytes(left) + token_to_bytes(right)

def decode_tokens(token_ids):
    data = b"".join(token_to_bytes(int(token_id)) for token_id in token_ids)
    return data.decode("utf-8", errors="ignore")

def encode_text(text):
    tokens = list(text.encode("utf-8", errors="ignore"))
    for new_token, pair in merge_rules:
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                merged.append(new_token)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        tokens = merged
    return tokens

def make_response(seed_text, response_text):
    return {"seed": seed_text, "response": response_text}


# ---- Transformer dataloaders ----
# Reuse the earlier train/validation split created above.
# The dataset objects were built from `input_sequences` and `target_tokens`.
# The transformer uses a dense collate function here rather than packed sequences.
def transformer_collate_fn(batch):
    xs, ys, _ = zip(*batch)
    return torch.stack(xs), torch.stack(ys)

tf_train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=transformer_collate_fn
)

tf_val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=transformer_collate_fn
)


# ---- Transformer model ----
class TransformerLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        nhead=8,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1,
        max_len=sequence_length,
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch_size_, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size_, seq_len)

        x = self.token_embedding(x) + self.position_embedding(positions)
        x = self.dropout(x)

        # This mask only allows the model to "see" tokens up until now
        # and not look ahead to "steal" answers.
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )

        x = self.transformer(x, mask=causal_mask)
        x = self.norm(x)
        logits = self.fc_out(x[:, -1, :])
        return logits


# ---- Training utilities ----
model = TransformerLanguageModel(vocab_size=len(vocab)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.set_grad_enabled(train):
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(dim=-1) == y).sum().item()
            total_count += x.size(0)

    return total_loss / total_count, total_correct / total_count

def train_transformer(num_epochs=3):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(tf_train_loader, train=True)
        val_loss, val_acc = run_epoch(tf_val_loader, train=False)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

    return history


# ---- Load or train ----
if not TRAIN_FROM_SCRATCH and os.path.exists(weight_path):
    checkpoint = torch.load(weight_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded weights from: {weight_path}")
    history = train_transformer(num_epochs=20)
else:
    history = train_transformer(num_epochs=20)
    if SAVE_WEIGHTS:
        os.makedirs(os.path.dirname(weight_path), exist_ok=True)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "vocab_size": len(vocab),
                "sequence_length": sequence_length,
            },
            weight_path,
        )
        print(f"Saved weights to: {weight_path}")


# ---- Text generation ----
def sample_next_token(logits, temperature=0.8, top_k=40):
    if temperature <= 0:
        return torch.argmax(logits, dim=-1).item()

    logits = logits / temperature

    if top_k is not None and top_k > 0:
        values, indices = torch.topk(logits, k=min(top_k, logits.size(-1)))
        probs = torch.softmax(values, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        return indices[next_idx].item()

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()

def get_response(seed_text, max_new_tokens=80, temperature=0.8, top_k=40):
    model.eval()

    token_ids = encode_text(seed_text)
    if len(token_ids) == 0:
        token_ids = encode_text(" ")

    generated = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = token_ids[-sequence_length:]
            x = torch.tensor(context, dtype=torch.long, device=DEVICE).unsqueeze(0)
            logits = model(x)[0]
            next_token = sample_next_token(logits, temperature=temperature, top_k=top_k)

            token_ids.append(next_token)
            generated.append(next_token)

    return make_response(seed_text, decode_tokens(generated))


# ---- Chat helper ----
def build_chat_prompt(previous_context, new_prompt):
    parts = []
    for item in previous_context:
        parts.append(f"User: {item['seed']}\nAssistant: {item['response']}")
    parts.append(f"User: {new_prompt}\nAssistant:")
    return "\n\n".join(parts)

def chat(previous_context, new_prompt, max_new_tokens=80, temperature=0.8, top_k=40):
    previous_context = [] if previous_context is None else list(previous_context)
    packaged_prompt = build_chat_prompt(previous_context, new_prompt)
    raw_result = get_response(
        packaged_prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
    )
    chat_entry = make_response(new_prompt, raw_result["response"])
    updated_context = previous_context + [chat_entry]
    return chat_entry, updated_context


Epoch 1/3 | train_loss=8.8693, train_acc=0.0035 | val_loss=8.6616, val_acc=0.0030
Epoch 2/3 | train_loss=8.3556, train_acc=0.0032 | val_loss=8.7681, val_acc=0.0017
Epoch 3/3 | train_loss=8.1313, train_acc=0.0053 | val_loss=8.7922, val_acc=0.0039
Saved weights to: /content/drive/MyDrive/Colab Notebooks/trans_whackGPT_weights.pth


In [7]:
# Example:
response = get_response("I need to confess that")
print(response)
#
context = []
reply, context = chat(context, "Hello, who are you?")
print(reply)

{'seed': 'I need to confess that', 'response': 'good since threw it not just s eI just  air we ingfor to conday but get tell himoverso dschool was ,gave .he ti the tohe was ve it was m I was for  always ins, the  and or she in I didn’t , for of his of the  and ed dmeto was , s, forupon of ,  a sher it ly his i were her never his very never in the so ed er and '}
{'seed': 'Hello, who are you?', 'response': 'for we s, ly ended in the hallthis was possible for it be .choic, coso onll *even pinpoint or ed away from felt chto just , and s and that s and svery ss and "in .to since my guy my just smy but  the upin showit .it  and sthe yhad )to just his  this eissto , and it very inkept to it was of she it  and  and us'}
